# Suspect X — Phase 1: Train the Interrogator (Qwen2.5-7B-Instruct + GRPO)

**Setup**: Colab A100 (40GB) recommended. T4 will OOM at k=8 generations; drop to k=4 if forced.

**Goal**: bring extraction rate on the 30 heldout crimes from ~11% (template baseline) to 40%+ in ~300 GRPO steps.

**What this trains**: a single LoRA adapter on top of frozen 4-bit Qwen2.5-7B-Instruct. The Suspect is the deterministic `RuleBasedSuspect` — *not* an LLM — for Phase 1. That fixes the adversary so any reward improvement is unambiguously attributable to the interrogator's policy.

## 1. Install deps and pull the env code

In [ ]:
%%capture
# Order matters: unsloth pulls in a pinned torch/triton/peft trio.
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.12.0" "peft>=0.13" "accelerate>=0.34" "bitsandbytes>=0.43"
!pip install "datasets>=2.20" "matplotlib" "pydantic>=2.6" "fastapi" "uvicorn" "httpx"

In [ ]:
# Bring the env code into the runtime. Two options:
#   (a) Mount the repo via Drive  (simplest)
#   (b) Clone from your GitHub fork once you push it
import os, sys
REPO_ROOT = "/content/metaFinale"   # <- adjust if needed
DESCRIPTIONS = f"{REPO_ROOT}/descriptions"

# Example (b):
# !git clone https://github.com/<you>/metaFinale.git $REPO_ROOT

assert os.path.isdir(DESCRIPTIONS), f"copy descriptions/ to {DESCRIPTIONS}"
sys.path.insert(0, REPO_ROOT)
os.environ["SUSPECT_X_DESCRIPTIONS"] = DESCRIPTIONS

## 2. Load Qwen2.5-7B-Instruct in 4-bit + attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

# T4: 16GB VRAM, no bf16 support. Trim seq length to fit + speed up.
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    dtype=None,            # autodetect: fp16 on T4
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
FastLanguageModel.for_training(model)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 3. Wire Qwen up as the InterrogatorFn

GRPOTrainer drives generation through its own internal sampler, but we also need a Python-level `interrogator_fn(messages) -> str` for our `run_episode` driver. The two paths share the same chat template.

In [ ]:
from suspect_x_env.training.rollout import run_episode
from suspect_x_env.training.rule_based_suspect import RuleBasedSuspect
from suspect_x_env.server.secret_factory import SecretFactory
from suspect_x_env.training.prompts import (
    interrogator_system_prompt, interrogator_user_turn, parse_accusation,
)

factory = SecretFactory()
print(f"loaded {len(factory)} crimes ({len(factory.split('train'))} train / {len(factory.split('heldout'))} heldout)")

GEN_KWARGS = dict(
    max_new_tokens=96,         # T4: shorter turns to save time
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    pad_token_id=tokenizer.eos_token_id,
)

def qwen_interrogator(messages):
    """InterrogatorFn matching rollout.py's signature."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KWARGS)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text


## 4. Sanity check: zero-shot rollout on one crime

In [ ]:
secret = factory.get("crime_001")
ep = run_episode(
    secret=secret,
    interrogator_fn=qwen_interrogator,
    suspect_fn=RuleBasedSuspect(secret, seed=0),
    max_turns=20,
)
print("extraction:", ep.grade.extraction_score)
print("reward:",     ep.grade.interrogator_reward)
print("matched:",    ep.grade.matched_keys)
print("accusation:", ep.accusation)
print("--- last 4 turns ---")
for t in ep.conversation[-4:]:
    print(f"[{t['role']}] {t['content'][:140]}")

## 5. Build the GRPO training data

GRPO needs a `Dataset` of prompts. For us a 'prompt' is a crime — the model's full episode is the 'completion' that gets graded. We use a thin custom rollout to do that.

In [ ]:
from datasets import Dataset

train_crimes = factory.split("train")
heldout_crimes = factory.split("heldout")

def crime_to_row(secret):
    return {
        "crime_id": secret.crime_id,
        # The 'prompt' field is what GRPOTrainer surfaces to reward_funcs.
        # We use the chat-template-rendered system prompt as the prompt.
        "prompt": interrogator_system_prompt(secret),
    }

train_ds = Dataset.from_list([crime_to_row(s) for s in train_crimes])
print(train_ds)

## 6. Reward function — full episode rollout per generation

Standard GRPO scores a single completion. We override that: each 'completion' is treated as a seed for a full 20-turn episode against the rule-based suspect, and the deterministic grader returns the reward.

In [ ]:
import re

def reward_episode(prompts, completions, crime_id=None, **kwargs):
    """GRPO reward function.

    Args mimic TRL's GRPOTrainer convention. `crime_id` is broadcast from
    the dataset column. We ignore the model's free-form `completion` text
    and instead run a full episode for grading — this gives the trainer a
    consistent, deterministic reward landscape.

    NOTE: This means the policy gradient acts on the FIRST-turn output
    only (which is what `completions` contains). Episode quality is the
    reward signal for that first turn. Phase 2 generalises to multi-turn
    credit assignment.
    """
    rewards = []
    crime_ids = crime_id if isinstance(crime_id, list) else [crime_id] * len(completions)
    for cid, _completion in zip(crime_ids, completions):
        secret = factory.get(cid)
        ep = run_episode(
            secret=secret,
            interrogator_fn=qwen_interrogator,
            suspect_fn=RuleBasedSuspect(secret, seed=0),
            max_turns=20,
        )
        rewards.append(float(ep.grade.interrogator_reward))
    return rewards

## 7. Configure and launch GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import statistics, json, os

EVAL_LOG_PATH = "./eval_log.jsonl"
HELDOUT_SUBSET = heldout_crimes[:5]   # T4: 5 crimes per eval (~3 min)

def _eval_now(label):
    extr, rew = [], []
    for i, secret in enumerate(HELDOUT_SUBSET):
        ep = run_episode(
            secret=secret,
            interrogator_fn=qwen_interrogator,
            suspect_fn=RuleBasedSuspect(secret, seed=i),
        )
        extr.append(ep.grade.extraction_score)
        rew.append(ep.grade.interrogator_reward)
    row = {"label": label, "mean_extr": statistics.mean(extr), "mean_reward": statistics.mean(rew)}
    print(f"[eval @ {label}] mean_extr={row['mean_extr']:.3f} mean_reward={row['mean_reward']:.3f}")
    with open(EVAL_LOG_PATH, "a") as f:
        f.write(json.dumps(row) + "\n")
    return row

class HeldoutEvalCallback(TrainerCallback):
    def __init__(self, every=20):
        self.every = every
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.every == 0:
            _eval_now(f"step_{state.global_step}")

if not os.path.exists(EVAL_LOG_PATH):
    _eval_now("step_0")

# T4 settings: fp16 (no bf16), k=2 (was 4), 80 steps (was 200), grad_accum=2
grpo_config = GRPOConfig(
    output_dir="./checkpoints/interrogator",
    num_generations=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_prompt_length=768,
    max_completion_length=96,
    learning_rate=5e-6,
    beta=0.04,
    max_steps=80,
    logging_steps=2,
    save_steps=40,
    report_to="tensorboard",
    warmup_ratio=0.1,
    fp16=True,                  # T4 cannot bf16
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_episode],
    args=grpo_config,
    train_dataset=train_ds,
    callbacks=[HeldoutEvalCallback(every=20)],
)
trainer.train()
_eval_now("step_final")


## 8. Periodic eval on heldout (run during/after training)

In [ ]:
import statistics

def evaluate(crimes, n=10, label=""):
    extr, rew = [], []
    for i, secret in enumerate(crimes[:n]):
        ep = run_episode(
            secret=secret,
            interrogator_fn=qwen_interrogator,
            suspect_fn=RuleBasedSuspect(secret, seed=i),
        )
        extr.append(ep.grade.extraction_score)
        rew.append(ep.grade.interrogator_reward)
    print(f"[{label}] n={n} mean_extr={statistics.mean(extr):.3f} mean_reward={statistics.mean(rew):.3f}")
    return extr, rew

evaluate(heldout_crimes, n=15, label="heldout @ end")

## 9. Save the LoRA adapter + plot the reward curve

In [ ]:
model.save_pretrained("./checkpoints/interrogator/lora_final")
tokenizer.save_pretrained("./checkpoints/interrogator/lora_final")
print("saved LoRA adapter")

In [ ]:
import matplotlib.pyplot as plt, json

log = trainer.state.log_history
steps = [e["step"] for e in log if "reward" in e]
rewards = [e["reward"] for e in log if "reward" in e]

eval_rows = [json.loads(l) for l in open("./eval_log.jsonl")]
eval_x = [int(r["label"].split("_")[1]) if r["label"] != "step_final" else 80 for r in eval_rows]
eval_y = [r["mean_extr"] for r in eval_rows]

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(steps, rewards, color="tab:blue")
ax[0].axhline(0.117, color="gray", ls="--", label="template baseline (heldout reward)")
ax[0].set_xlabel("GRPO step"); ax[0].set_ylabel("mean group reward")
ax[0].set_title("Phase 1 — interrogator training reward (T4, 80 steps)")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(eval_x, eval_y, marker="o", color="tab:green", label="trained Qwen (heldout)")
ax[1].axhline(0.111, color="gray", ls="--", label="template baseline 11.1%")
ax[1].axhline(0.000, color="lightgray", ls=":", label="random baseline 0%")
ax[1].set_xlabel("GRPO step"); ax[1].set_ylabel("mean extraction rate")
ax[1].set_title("Heldout extraction (n=5 unseen crimes)")
ax[1].set_ylim(-0.02, 1.0); ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()
print("saved reward_curve.png")
